In [1]:
import pandas as pd

In [2]:
# Load the standard Adult Income dataset (UCI)
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
column_names = [
    'age', 'workclass', 'fnlwgt', 'education', 'education-num',
    'marital-status', 'occupation', 'relationship', 'race', 'sex',
    'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income'
]

In [3]:
real_data = pd.read_csv(url, names=column_names, na_values=' ?', skipinitialspace=True)
real_data = real_data.dropna()  # remove rows with missing values (standard practice)

In [4]:
print("✅ Dataset loaded successfully!")
print(f"Shape: {real_data.shape}")
print(real_data['income'].value_counts())
print(real_data['sex'].value_counts())

✅ Dataset loaded successfully!
Shape: (32561, 15)
income
<=50K    24720
>50K      7841
Name: count, dtype: int64
sex
Male      21790
Female    10771
Name: count, dtype: int64


In [7]:
!pip install sdv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.6/202.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 88.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.3/202.3 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.9 MB/s eta 0:00:00


In [8]:
from sdv.single_table import CTGANSynthesizer, TVAESynthesizer
from sdv.metadata import SingleTableMetadata
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

In [9]:
# ====================== 1. Metadata ======================
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(real_data)
print("✅ Metadata detected")

✅ Metadata detected


In [10]:
# ====================== 2. Train CTGAN ======================
print("\n🚀 Training CTGAN...")
ctgan = CTGANSynthesizer(metadata, epochs=300, batch_size=500, verbose=True)
ctgan.fit(real_data)


🚀 Training CTGAN...


Gen. (-00.49) | Discrim. (-00.16): 100%|██████████| 300/300 [1:02:20<00:00, 12.47s/it]


In [11]:
# Generate regular synthetic data
synthetic_ctgan = ctgan.sample(num_rows=len(real_data))

In [12]:
# ====================== 3. Train TVAE ======================
print("\n🚀 Training TVAE...")
tvae = TVAESynthesizer(metadata, epochs=300, batch_size=500, verbose=True)
tvae.fit(real_data)
synthetic_tvae = tvae.sample(num_rows=len(real_data))


🚀 Training TVAE...


Loss: -15.12: 100%|██████████| 300/300 [29:47<00:00,  5.96s/it]


In [19]:
# ====================== 4. Bias-Mitigated Synthetic Data (FIXED) ======================
print("\n⚖️  Generating bias-mitigated synthetic data using CTGAN...")

N = len(real_data) // 2

# Generate a sufficiently large synthetic dataset first
# We'll oversample by a factor (e.g., 3) to ensure enough samples are available for both conditions
oversample_factor = 3
total_synthetic_samples_needed = 2 * N
initial_sample_size = total_synthetic_samples_needed * oversample_factor

print(f"Generating an initial synthetic dataset of {initial_sample_size} rows...")
synthetic_data_unfiltered = ctgan.sample(num_rows=initial_sample_size)

# Filter for high income and take N samples
high_income_mask = synthetic_data_unfiltered['income'] == '>50K'
available_high_income = synthetic_data_unfiltered[high_income_mask]
# Ensure we don't try to sample more rows than available
balanced_synthetic_high_income = available_high_income.sample(n=min(N, len(available_high_income)), random_state=42)

# Filter for low income and take N samples
low_income_mask = synthetic_data_unfiltered['income'] == '<=50K'
available_low_income = synthetic_data_unfiltered[low_income_mask]
# Ensure we don't try to sample more rows than available
balanced_synthetic_low_income = available_low_income.sample(n=min(N, len(available_low_income)), random_state=42)

# Concatenate the balanced samples
balanced_synthetic = pd.concat([balanced_synthetic_high_income, balanced_synthetic_low_income], ignore_index=True)

print(f"✅ Balanced synthetic shape: {balanced_synthetic.shape}")
print("Income distribution in balanced data:\n", balanced_synthetic['income'].value_counts())


⚖️  Generating bias-mitigated synthetic data using CTGAN...
Generating an initial synthetic dataset of 97680 rows...
✅ Balanced synthetic shape: (32560, 15)
Income distribution in balanced data:
 income
>50K     16280
<=50K    16280
Name: count, dtype: int64


In [20]:
# ====================== 5. Save Files ======================
synthetic_ctgan.to_csv('synthetic_ctgan.csv', index=False)
synthetic_tvae.to_csv('synthetic_tvae.csv', index=False)
balanced_synthetic.to_csv('synthetic_balanced.csv', index=False)
print("✅ All synthetic datasets saved!")

✅ All synthetic datasets saved!


In [21]:
# ====================== 6. Evaluation Function ======================
def evaluate_model(train_df, test_df, name):
    X_train = pd.get_dummies(train_df.drop(columns=['income']), drop_first=True)
    y_train = (train_df['income'] == '>50K').astype(int)

    X_test = pd.get_dummies(test_df.drop(columns=['income']), drop_first=True)
    y_test = (test_df['income'] == '>50K').astype(int)
    X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

    # Logistic Regression
    lr = LogisticRegression(max_iter=1000)
    lr.fit(X_train, y_train)
    lr_acc = accuracy_score(y_test, lr.predict(X_test))

    # XGBoost
    xgb_model = xgb.XGBClassifier(eval_metric='logloss', random_state=42)
    xgb_model.fit(X_train, y_train)
    xgb_acc = accuracy_score(y_test, xgb_model.predict(X_test))

    # Fairness metric
    test_df = test_df.copy()
    test_df['pred'] = xgb_model.predict(X_test)
    male_rate = test_df[test_df['sex'] == 'Male']['pred'].mean()
    female_rate = test_df[test_df['sex'] == 'Female']['pred'].mean()
    parity_gap = abs(male_rate - female_rate)

    print(f"\n📊 {name} Results:")
    print(f"   LR Accuracy      : {lr_acc:.4f}")
    print(f"   XGBoost Accuracy : {xgb_acc:.4f}")
    print(f"   Parity Gap       : {parity_gap:.4f}")
    return xgb_acc, parity_gap

In [22]:
# ====================== 7. Run Evaluation ======================
train_real, test_real = train_test_split(real_data, test_size=0.2, random_state=42)

print("\n" + "="*60)
print("BASELINE - Real Data")
evaluate_model(train_real, test_real, "Real Data")

print("\n" + "="*60)
print("CTGAN Synthetic")
evaluate_model(synthetic_ctgan, test_real, "CTGAN")

print("\n" + "="*60)
print("TVAE Synthetic")
evaluate_model(synthetic_tvae, test_real, "TVAE")

print("\n" + "="*60)
print("BALANCED Synthetic (Bias-Mitigated)")
evaluate_model(balanced_synthetic, test_real, "Balanced Synthetic")

print("\n🎉 Project completed successfully!")


BASELINE - Real Data

📊 Real Data Results:
   LR Accuracy      : 0.8475
   XGBoost Accuracy : 0.8739
   Parity Gap       : 0.1815

CTGAN Synthetic

📊 CTGAN Results:
   LR Accuracy      : 0.8268
   XGBoost Accuracy : 0.8365
   Parity Gap       : 0.1637

TVAE Synthetic

📊 TVAE Results:
   LR Accuracy      : 0.8263
   XGBoost Accuracy : 0.8254
   Parity Gap       : 0.2680

BALANCED Synthetic (Bias-Mitigated)

📊 Balanced Synthetic Results:
   LR Accuracy      : 0.7643
   XGBoost Accuracy : 0.7990
   Parity Gap       : 0.2789

🎉 Project completed successfully!
